In [18]:
import os

import pandas as pd
from graphrag.query.context_builder.entity_extraction import EntityVectorStoreKey
from graphrag.query.indexer_adapters import (
    read_indexer_entities,
    read_indexer_relationships,
    read_indexer_reports,
    read_indexer_text_units,
)
from graphrag.query.structured_search.local_search.mixed_context import (
    LocalSearchMixedContext,
)
from graphrag.query.structured_search.local_search.search import LocalSearch
from graphrag_vectors import IndexSchema
from graphrag_vectors.lancedb import LanceDBVectorStore

import dotenv
dotenv.load_dotenv()

True

## Local Search Example

Local search method generates answers by combining relevant data from the AI-extracted knowledge-graph with text chunks of the raw documents. This method is suitable for questions that require an understanding of specific entities mentioned in the documents (e.g. What are the healing properties of chamomile?).

### Load text units and graph data tables as context for local search

- In this test we first load indexing outputs from parquet files to dataframes, then convert these dataframes into collections of data objects aligning with the knowledge model.

### Load tables to dataframes

In [19]:
INPUT_DIR = ".workspace/515b536d-ab6f-4c9c-9e8e-caf2147d0aed/VBS/output"
LANCEDB_URI = f"{INPUT_DIR}/lancedb"

COMMUNITY_REPORT_TABLE = "community_reports"
ENTITY_TABLE = "entities"
COMMUNITY_TABLE = "communities"
RELATIONSHIP_TABLE = "relationships"
COVARIATE_TABLE = "covariates"
TEXT_UNIT_TABLE = "text_units"
COMMUNITY_LEVEL = 2

#### Read entities

In [20]:
# read nodes table to get community and degree data
entity_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_TABLE}.parquet")
community_df = pd.read_parquet(f"{INPUT_DIR}/{COMMUNITY_TABLE}.parquet")

entities = read_indexer_entities(entity_df, community_df, COMMUNITY_LEVEL)


description_embedding_store = LanceDBVectorStore(
    db_uri=LANCEDB_URI,
    index_name="entity_description"
)

description_embedding_store.connect()

print(f"Entity count: {len(entity_df)}")
entity_df.head()

Entity count: 154


,id,human_readable_id,title,type,description,text_unit_ids,frequency,degree
0,97bf299d-b754-4549-8ab2-1c9d36eb0aaa,0,USER,ROLE,"The entity ""USER"" represents various individua...",[a819b6134907c3304b71ee4962f872e3f58c86f66f4a3...,20,17
1,4bfb11e3-62f4-4218-9cac-321c91dfabe9,1,INTERNATIONAL AIRPORT,GEO,The specific geographic location for airport t...,[a819b6134907c3304b71ee4962f872e3f58c86f66f4a3...,2,2
2,414deb1d-7070-450d-a8f1-761a31fdab4c,2,FIXED PRICE OF $30,VALUE,A predetermined price for airport trips,[a819b6134907c3304b71ee4962f872e3f58c86f66f4a3...,2,4
3,9e0e4d67-08b2-4ff2-ab52-371f0b129280,3,AIRPORT TRIP,EVENT,A trip with the destination being an airport,[a819b6134907c3304b71ee4962f872e3f58c86f66f4a3...,2,2
4,e771d5c3-4cfe-4f7e-b34a-72ad114a0d8c,4,TRAFFIC,RESOURCE,An external factor that can influence travel t...,[a819b6134907c3304b71ee4962f872e3f58c86f66f4a3...,2,1


#### Read relationships

In [21]:
relationship_df = pd.read_parquet(f"{INPUT_DIR}/{RELATIONSHIP_TABLE}.parquet")
relationships = read_indexer_relationships(relationship_df)

print(f"Relationship count: {len(relationship_df)}")
relationship_df.head()

Relationship count: 176


,id,human_readable_id,source,target,description,weight,combined_degree,text_unit_ids
0,62f5088e-4017-4d4a-bef7-b2e5fe54477c,0,USER,FIXED PRICE OF $30,The user desires a fixed price for airport trips,20.0,21,[a819b6134907c3304b71ee4962f872e3f58c86f66f4a3...
1,6237424b-3679-48d4-a95d-aadba0a476a6,1,AIRPORT TRIP,INTERNATIONAL AIRPORT,Airport trips are defined by the destination b...,18.0,4,[a819b6134907c3304b71ee4962f872e3f58c86f66f4a3...
2,f7a3f4e6-3fdc-4bc9-b8d2-cd58a95311c1,2,FIXED PRICE OF $30,AIRPORT TRIP,The fixed price applies specifically to airpor...,20.0,6,[a819b6134907c3304b71ee4962f872e3f58c86f66f4a3...
3,56265e88-5afb-4434-97d9-870db81a7490,3,FIXED PRICE OF $30,TRAFFIC,The fixed price makes the cost independent of ...,16.0,5,[a819b6134907c3304b71ee4962f872e3f58c86f66f4a3...
4,603c9711-3c25-43fb-9695-f88e2b7a127b,4,FIXED PRICE OF $30,DISTANCE WITHIN THE CITY,The fixed price makes the cost independent of ...,16.0,5,[a819b6134907c3304b71ee4962f872e3f58c86f66f4a3...


#### Read community reports

In [22]:
report_df = pd.read_parquet(f"{INPUT_DIR}/{COMMUNITY_REPORT_TABLE}.parquet")
reports = read_indexer_reports(report_df, community_df, COMMUNITY_LEVEL)

print(f"Report records: {len(report_df)}")
report_df.head()

Report records: 27


,id,human_readable_id,community,level,parent,children,title,summary,full_content,rank,rating_explanation,findings,full_content_json,period,size
0,fae2d6654e1949a2c293b7186030349e5834f75d061900...,15,15,1,0,[],Passenger Ride Cancellation Feature,This feature cluster focuses on the 'CANCEL RI...,# Passenger Ride Cancellation Feature\n\nThis ...,4.0,The complexity is moderate due to the clear de...,[{'explanation': 'The 'CANCEL RIDE WITHOUT FEE...,"{\n ""title"": ""Passenger Ride Cancellation F...",2026-04-08,6
1,07d94a30ba8c4a78c13649642e3e27b0f0e726b3a20892...,16,16,1,0,[],Cancellation Fee Processing,This cluster defines the functionality for cha...,# Cancellation Fee Processing\n\nThis cluster ...,4.0,"The current scope is well-defined but limited,...",[{'explanation': 'The 'CANCELLATION FEE' direc...,"{\n ""title"": ""Cancellation Fee Processing"",...",2026-04-08,2
2,2dd9c962dd023ab31be3ed61cb3c9e78b4456f178fe1d3...,17,17,1,0,[],Ride Request and Notification System Enhancements,This cluster of user stories focuses on enhanc...,# Ride Request and Notification System Enhance...,7.5,The high number of relationships converging on...,[{'explanation': 'Multiple functionalities are...,"{\n ""title"": ""Ride Request and Notification...",2026-04-08,5
3,9801fb41edc70205db0d433cb2f60b6d5d9a3a080a0328...,18,18,1,0,[],Ride Cancellation and Fee Waive,This community of entities focuses on the scen...,# Ride Cancellation and Fee Waive\n\nThis comm...,2.0,The risk and complexity are low due to the cle...,[{'explanation': 'There is a direct and sequen...,"{\n ""title"": ""Ride Cancellation and Fee Wai...",2026-04-08,3
4,4b69edbceb52af65602ec34ec82bdfafad30ea8dbd89e6...,19,19,1,5,[],Image Tagging in Asset Intelligence,This cluster focuses on the 'ENABLE IMAGE TAGG...,# Image Tagging in Asset Intelligence\n\nThis ...,7.0,"The rating is high due to the strong, critical...",[{'explanation': 'The 'ENABLE IMAGE TAGGING' f...,"{\n ""title"": ""Image Tagging in Asset Intell...",2026-04-08,4


#### Read text units

In [23]:
text_unit_df = pd.read_parquet(f"{INPUT_DIR}/{TEXT_UNIT_TABLE}.parquet")
text_units = read_indexer_text_units(text_unit_df)

# print(f"Text unit records: {len(text_unit_df)}")
text_unit_df.head()

,id,human_readable_id,text,n_tokens,document_id,entity_ids,relationship_ids,covariate_ids
0,a819b6134907c3304b71ee4962f872e3f58c86f66f4a35...,0,SUMMARY:\nFixed Price for All Airport Trips\n\...,83,a819b6134907c3304b71ee4962f872e3f58c86f66f4a35...,"[97bf299d-b754-4549-8ab2-1c9d36eb0aaa, 97bf299...","[62f5088e-4017-4d4a-bef7-b2e5fe54477c, 62f5088...",[]
1,bc78bd86bdc2c2c81085a0ce7eaa12c69ca725af035f0a...,1,SUMMARY:\nUpdate User Payment Info\n\nDESCRIPT...,78,bc78bd86bdc2c2c81085a0ce7eaa12c69ca725af035f0a...,"[97bf299d-b754-4549-8ab2-1c9d36eb0aaa, 90a4b83...","[709e8b56-3408-4a10-b856-8cfbd3a64088, 7958e7e...",[]
2,47505777d4c4088f9f0363c1438997447d69adf8264b26...,2,SUMMARY:\nMake the App Run Faster\n\nDESCRIPTI...,70,47505777d4c4088f9f0363c1438997447d69adf8264b26...,"[97bf299d-b754-4549-8ab2-1c9d36eb0aaa, 6aa8897...","[3b0d92f6-1842-4d40-8fcc-af646af01729, 5eaed68...",[]
3,7393290c6e2c61d8771b92cec90c77811bc47847cd8af3...,3,SUMMARY:\nBook a Ride for a Friend\n\nDESCRIPT...,90,7393290c6e2c61d8771b92cec90c77811bc47847cd8af3...,"[97bf299d-b754-4549-8ab2-1c9d36eb0aaa, 364e5fa...","[01499574-21d4-4ddd-b7a2-3899dd35691c, 3e70b79...",[]
4,5d691b98cd85d00ac3176bd0e7caccc0abd67bb07ff0e8...,4,SUMMARY:\nIn-App Chat with Driver\n\nDESCRIPTI...,74,5d691b98cd85d00ac3176bd0e7caccc0abd67bb07ff0e8...,"[97bf299d-b754-4549-8ab2-1c9d36eb0aaa, 90a4b83...","[709e8b56-3408-4a10-b856-8cfbd3a64088, 7958e7e...",[]


In [24]:
documents_df = pd.read_parquet(f"{INPUT_DIR}/documents.parquet")
print(f"Document records: {len(documents_df)}")
documents_df.head()

Document records: 20


,id,human_readable_id,title,text,text_unit_ids,creation_date,raw_data
0,a819b6134907c3304b71ee4962f872e3f58c86f66f4a35...,0,Fixed Price for All Airport Trips,SUMMARY:\nFixed Price for All Airport Trips\n\...,[a819b6134907c3304b71ee4962f872e3f58c86f66f4a3...,2026-04-08 14:05:50 +0700,{'description': 'SUMMARY: Fixed Price for All ...
1,bc78bd86bdc2c2c81085a0ce7eaa12c69ca725af035f0a...,1,Update User Payment Info,SUMMARY:\nUpdate User Payment Info\n\nDESCRIPT...,[bc78bd86bdc2c2c81085a0ce7eaa12c69ca725af035f0...,2026-04-08 14:05:50 +0700,{'description': 'SUMMARY: Update User Payment ...
2,47505777d4c4088f9f0363c1438997447d69adf8264b26...,2,Make the App Run Faster,SUMMARY:\nMake the App Run Faster\n\nDESCRIPTI...,[47505777d4c4088f9f0363c1438997447d69adf8264b2...,2026-04-08 14:05:50 +0700,{'description': 'SUMMARY: Make the App Run Fas...
3,7393290c6e2c61d8771b92cec90c77811bc47847cd8af3...,3,Book a Ride for a Friend,SUMMARY:\nBook a Ride for a Friend\n\nDESCRIPT...,[7393290c6e2c61d8771b92cec90c77811bc47847cd8af...,2026-04-08 14:05:50 +0700,{'description': 'SUMMARY: Book a Ride for a Fr...
4,5d691b98cd85d00ac3176bd0e7caccc0abd67bb07ff0e8...,4,In-App Chat with Driver,SUMMARY:\nIn-App Chat with Driver\n\nDESCRIPTI...,[5d691b98cd85d00ac3176bd0e7caccc0abd67bb07ff0e...,2026-04-08 14:05:50 +0700,{'description': 'SUMMARY: In-App Chat with Dri...


In [25]:
from graphrag_llm.completion.completion_factory import create_completion, create_tokenizer
from graphrag_llm.embedding.embedding_factory import create_embedding
from graphrag_llm.config import ModelConfig, TokenizerConfig

api_key = os.getenv("GRAPHRAG_MODEL_API_KEY")

print("API Key loaded:", api_key is not None)

chat_config = ModelConfig(
    api_key=api_key,
    model_provider="gemini",
    # model="gemma-4-31b-it",
    model="gemini-2.5-flash-lite"
)

tokenizer_config = TokenizerConfig(
    model_id="gemini/gemini-2.5-flash-lite"
)

tokenizer = create_tokenizer(tokenizer_config)

chat_model = create_completion(model_config=chat_config)

embedder_config = ModelConfig(
    api_key=api_key,
    model_provider="gemini",
    model="gemini-embedding-001",
)

text_embedder = create_embedding(
    model_config=embedder_config
)

API Key loaded: True


### Create local search context builder

In [26]:
context_builder = LocalSearchMixedContext(
    community_reports=reports,
    text_units=text_units,
    entities=entities,
    relationships=relationships,
    # if you did not run covariates during indexing, set this to None
    entity_text_embeddings=description_embedding_store,
    embedding_vectorstore_key=EntityVectorStoreKey.ID,  # if the vectorstore uses entity title as ids, set this to EntityVectorStoreKey.TITLE
    text_embedder=text_embedder,
    tokenizer=tokenizer,
    
)

### Create local search engine

In [27]:
# text_unit_prop: proportion of context window dedicated to related text units
# community_prop: proportion of context window dedicated to community reports.
# The remaining proportion is dedicated to entities and relationships. Sum of text_unit_prop and community_prop should be <= 1
# conversation_history_max_turns: maximum number of turns to include in the conversation history.
# conversation_history_user_turns_only: if True, only include user queries in the conversation history.
# top_k_mapped_entities: number of related entities to retrieve from the entity description embedding store.
# top_k_relationships: control the number of out-of-network relationships to pull into the context window.
# include_entity_rank: if True, include the entity rank in the entity table in the context window. Default entity rank = node degree.
# include_relationship_weight: if True, include the relationship weight in the context window.
# include_community_rank: if True, include the community rank in the context window.
# return_candidate_context: if True, return a set of dataframes containing all candidate entity/relationship/covariate records that
# could be relevant. Note that not all of these records will be included in the context window. The "in_context" column in these
# dataframes indicates whether the record is included in the context window.
# max_tokens: maximum number of tokens to use for the context window.


local_context_params = {
    "text_unit_prop": 0.5,
    "community_prop": 0.1,
    "conversation_history_max_turns": 5,
    "conversation_history_user_turns_only": True,
    "top_k_mapped_entities": 10,
    "top_k_relationships": 10,
    "include_entity_rank": True,
    "include_relationship_weight": True,
    "include_community_rank": False,
    "return_candidate_context": False,
    "embedding_vectorstore_key": EntityVectorStoreKey.ID,  # set this to EntityVectorStoreKey.TITLE if the vectorstore uses entity title as ids
    "max_tokens": 12_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 5000)
}

model_params = {
    "max_tokens": 2_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 1000=1500)
    "temperature": 0.0,
}

In [28]:
search_engine = LocalSearch(
    model=chat_model,
    context_builder=context_builder,
    tokenizer=tokenizer,
    model_params=model_params,
    context_builder_params=local_context_params,
    response_type="multiple paragraphs",  # free form text describing the response type and format, can be anything, e.g. prioritized list, single paragraph, multiple paragraphs, multiple-page report
)

### Run local search on sample queries

In [29]:
result = await search_engine.search("""
Is there any story about defining app UI?
Return a single JSON object with the following format:
{
    "stories": [
        {
            "explanation": <explanation>,
            "data": {
                "entities": [<list of relevant entities ID from the context>],
                "relationships": [<list of relevant relationships ID from the context>],
                "communityReports": [<list of relevant community reports ID from the context>],
                "sources": [<list of relevant text units ID from the context>]
            ]
        }
    ]
}
""")
print(result.response)

```json
{
    "stories": [
        {
            "explanation": "The application's user interface (UI) is designed with several features that enhance user experience and functionality. These include a dark mode option for comfortable night-time use, a chat overlay for in-app messaging with drivers, and a wallet screen for managing funds. The app also features a 'My Trips' section that displays trip history and allows for receipt downloads. Additionally, there are specific UI elements like the 'Open Home Screen' and 'Dark Mode Toggle' that initiate or modify the app's appearance and functionality.",
            "data": {
                "entities": [
                    "34",
                    "46",
                    "10",
                    "120",
                    "123",
                    "88",
                    "80",
                    "101",
                    "102",
                    "103"
                ],
                "relationships": [
                    "40"

#### Inspecting the context data used to generate the response

In [30]:
result.context_data["entities"].head()

,id,entity,description,number of relationships,in_context
0,53,SYSTEM,The provided descriptions consistently refer t...,12,True
1,88,OPEN HOME SCREEN,The action of launching the application's main...,1,True
2,102,USING THE APP,The act of interacting with the application,2,True
3,21,PERFORM ANY ACTION,Any interaction a user takes within the applic...,4,True
4,140,APP,The application through which the user sends m...,6,True


In [31]:
result.context_data["relationships"].head()

,id,source,target,description,weight,links,in_context
0,20,USER,PERFORM ANY ACTION,The user initiates and performs actions within...,10.0,1,True
1,139,USER,VIEW TRIP HISTORY,The user performs the action of viewing trip h...,20.0,2,True
2,143,USER,MY TRIPS,The user navigates to the 'My Trips' section,18.0,1,True
3,25,APP,PERFORM ANY ACTION,Actions are performed within the application,8.0,2,True
4,24,APP,APP IS OPEN,The application needs to be open for any actio...,9.0,2,True


In [32]:
if "reports" in result.context_data:
    result.context_data["reports"].head()

In [33]:
result.context_data["sources"].head()

,id,text
0,9,SUMMARY:\nStrict 15-Minute Cancellation Grace ...
1,7,SUMMARY:\nPush Notification for Driver Arrival...
2,11,SUMMARY:\nReal-time Vehicle Search by Location...
3,13,SUMMARY:\nSystem Logic: Do the Right Thing\n\n...
4,2,SUMMARY:\nMake the App Run Faster\n\nDESCRIPTI...


In [34]:
if "claims" in result.context_data:
    print(result.context_data["claims"].head())